### Bronze (ingest)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/my_files/naijamart/")

In [0]:
customer_schema = StructType([
  StructField("customer_id", StringType(), True),
  StructField("full_name", StringType(), True),
  StructField("email", StringType(), True),
  StructField("phone", StringType(), True),
  StructField("city", StringType(), True),
  StructField("signup_date", StringType(), True)
])
order_schema = StructType([
  StructField("order_id", StringType(), True),
  StructField("customer_id", StringType(), True),
  StructField("product_id", StringType(), True),
  StructField("quantity", StringType(), True),
  StructField("discount_pct", StringType(), True),
  StructField("order_date", StringType(), True),
  StructField("payment_method", StringType(), True),
  StructField("order_status", StringType(), True)
])
product_schema = StructType([
  StructField("category", StringType(), True),
  StructField("product_id", StringType(), True),
  StructField("product_name", StringType(), True),
  StructField("supplier", StringType(), True),
  StructField("unit_price", StringType(), True)
])

In [0]:
customers = spark.read.format("csv").schema(customer_schema).option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/default/my_files/naijamart/naijamart_customers.csv").select("*", col("_metadata.file_name").alias("_source_file"))
orders = spark.read.format("csv").option("header", "true").schema(order_schema).option("inferSchema", "true").load("/Volumes/workspace/default/my_files/naijamart/naijamart_orders.csv").select("*", col("_metadata.file_name").alias("_source_file"))
products = spark.read.format("json").option("header", "true").schema(product_schema).option("inferSchema", "true").load("/Volumes/workspace/default/my_files/naijamart/naijamart_products.json").select("*", col("_metadata.file_name").alias("_source_file"))

In [0]:
customers = customers.withColumn("_ingested_at", current_timestamp())
orders = orders.withColumn("_ingested_at", current_timestamp())
products = products.withColumn("_ingested_at", current_timestamp())
customers.limit(10).display()
orders.limit(10).display()
products.limit(10).display()

In [0]:
customers.write.mode("overwrite").saveAsTable("customers")
orders.write.mode("overwrite").saveAsTable("orders")
products.write.mode("overwrite").saveAsTable("products")

### Silver (clean, dedupe, validate)

In [0]:
string_columns = [name for name, dtype in customers.dtypes if dtype == "string"]

customers = customers.withColumns({
    name: when(trim(col(name)) == "", None).otherwise(trim(col(name)))
    for name in string_columns
})